<a href="https://colab.research.google.com/github/AgrimaOjha/ARIS/blob/main/ARIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    f1_score
)

# Model
from xgboost import XGBClassifier


In [4]:
df = pd.read_csv("data.csv")

In [5]:
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [8]:
df.nunique()

,0
customerID,7043
gender,2
SeniorCitizen,2
Partner,2
Dependents,2
tenure,73
PhoneService,2
MultipleLines,3
InternetService,3
OnlineSecurity,3


In [9]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [10]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [11]:
df.drop('customerID', axis=1, inplace=True)

In [12]:
df['Churn'] = df['Churn'].map({'No':0, 'Yes':1})

In [13]:
df['Tenure_Group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=[0,1,2,3]
)

In [14]:
df['Charge_per_Tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

In [15]:
df = pd.get_dummies(df, drop_first=True)


In [16]:
df

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,Charge_per_Tenure,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Tenure_Group_1,Tenure_Group_2,Tenure_Group_3
0,0,1,29.85,29.85,0,14.925000,False,True,False,False,...,False,False,False,True,False,True,False,False,False,False
1,0,34,56.95,1889.50,0,1.627143,True,False,False,True,...,False,True,False,False,False,False,True,False,True,False
2,0,2,53.85,108.15,1,17.950000,True,False,False,True,...,False,False,False,True,False,False,True,False,False,False
3,0,45,42.30,1840.75,0,0.919565,True,False,False,False,...,False,True,False,False,False,False,False,False,True,False
4,0,2,70.70,151.65,1,23.566667,False,False,False,True,...,False,False,False,True,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,0,3.392000,True,True,True,True,...,True,True,False,True,False,False,True,True,False,False
7039,0,72,103.20,7362.90,0,1.413699,False,True,True,True,...,True,True,False,True,True,False,False,False,False,True
7040,0,11,29.60,346.45,0,2.466667,False,True,True,False,...,False,False,False,True,False,True,False,False,False,False
7041,1,4,74.40,306.60,1,14.880000,True,True,False,True,...,False,False,False,True,False,False,True,False,False,False


In [17]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [18]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

In [19]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
y_prob = xgb.predict_proba(X_test)[:,1]

In [21]:
thresholds = np.arange(0.30, 0.70, 0.02)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    y_pred_temp = (y_prob > t).astype(int)
    score = f1_score(y_test, y_pred_temp)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

# Final predictions
y_pred = (y_prob > best_threshold).astype(int)

Best Threshold: 0.5200000000000002
Best F1 Score: 0.6215921483097055


In [22]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

tn, fp, fn, tp = cm.ravel()

precision = tp / (tp + fp)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = 2 * precision * recall / (precision + recall)

print("\nAdditional Metrics:")
print("Precision:", precision)
print("Recall:", recall)
print("Specificity:", specificity)
print("F1 Score:", f1)


Accuracy: 0.7537260468417317
ROC-AUC: 0.8364450127877238

Confusion Matrix:
 [[777 258]
 [ 89 285]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.52      0.76      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.77      1409


Additional Metrics:
Precision: 0.5248618784530387
Recall: 0.7620320855614974
Specificity: 0.7507246376811594
F1 Score: 0.6215921483097055


In [23]:
import joblib

joblib.dump(xgb, "churn_model.pkl")
joblib.dump(X.columns.tolist(), "model_columns.pkl")

['model_columns.pkl']

In [24]:
from google.colab import files
files.download("churn_model.pkl")
files.download("model_columns.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# MILESTONE 2

In [ ]:
!pip install -q langgraph langchain_groq langchain_community faiss-cpu sentence-transformers
!pip install -q --upgrade langchain-huggingface langgraph langchain-groq langchain-community faiss-cpu sentence-transformers
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 33.7 MB/s eta 0:00:00


In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get('groq_api')

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
# 1. Define your Knowledge Base (The 'Strategy Playbook')
retention_docs = [
    "KB-ID: TELCO-001 | Category: Infrastructure | Issue: Fiber Optic Signal Instability. "
    "Protocol: Trigger technical diagnostic. Incentive: Service Credit (1-month) for service interruption mitigation.",

    "KB-ID: TELCO-002 | Category: Contractual | Issue: Short-term (M2M) Volatility. "
    "Protocol: Commitment Escalation. Incentive: 15% ARR (Annual Recurring Revenue) reduction upon 12-month renewal.",

    "KB-ID: TELCO-003 | Category: Billing | Issue: Transactional Friction (Electronic Check). "
    "Protocol: Autopay Conversion. Incentive: $10.00 'Digital Adoption' one-time credit.",

    "KB-ID: TELCO-004 | Category: Revenue Management | Issue: ARPU (Average Revenue Per User) Threshold Breach. "
    "Protocol: Down-sell to Tier-2 'Value-Plus' bundle to protect retention over margin."
]

# 2. Vectorize using free HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_texts(retention_docs, embeddings)
retriever = vector_db.as_retriever()

In [ ]:
# 1. Define the State Schema (Includes history for 'Confusions' & Interactive chat)
class AgentState(TypedDict):
    customer_id: str
    churn_risk: str # Changed to string for prettier formatting
    important_features: List[str]
    retrieved_strategies: str
    chat_history: List[str]
    user_query: str
    final_recommendation: str

# 2. Node 1: Intelligent Retrieval
def retrieve_node(state: AgentState):
    # This searches your FAISS DB for relevant strategies
    query = f"Strategies for {', '.join(state['important_features'])}"
    docs = retriever.invoke(query)
    return {"retrieved_strategies": "\n".join([d.page_content for d in docs])}

# 3. Node 2: The Master Strategist Reasoning
def interactive_planner(state: AgentState):
    # The prompt you wanted: Detailed, clear, and professional
    rag_system_prompt = """
    ### ROLE
    You are the **Lead Agentic Retention Strategist**. Your goal is to convert predictive churn data into high-impact, empathetic, and economically viable retention plans.

    ### OPERATIONAL CONTEXT
    - **Target Customer Churn Risk:** {risk}
    - **Primary Churn Drivers:** {drivers}
    - **Retrieved Strategy Guidelines:** {context}

    ### TASK INSTRUCTIONS
    1. **Analyze with Depth:** Explain the psychology behind the churn drivers.
    2. **Consultative Strategy:** Only recommend strategies from the guidelines that solve this specific customer's issues.
    3. **Economic Guardrails:** Ensure the "Discount/Offer" is proportional to the risk score.
    4. **State Management:** Reference the chat history if the user is asking a follow-up or clarifying a 'confusion'.

    ### STRUCTURED OUTPUT FORMAT
    #### 1. Executive Summary
    #### 2. Root Cause Analysis (The 'Why')
    #### 3. Agentic Intervention Plan
    #### 4. The 'Retention Pitch' (Script for Support Agent)
    #### 5. Compliance & Ethics (Disclaimers)
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", rag_system_prompt),
        ("placeholder", "{chat_history}"),
        ("user", "{user_query}")
    ])

    chain = prompt | llm

    response = chain.invoke({
        "risk": state['churn_risk'],
        "drivers": state['important_features'],
        "context": state['retrieved_strategies'],
        "chat_history": state.get('chat_history', []),
        "user_query": state['user_query']
    })

    return {"final_recommendation": response.content}



In [ ]:

# 4. Compile the Graph
builder = StateGraph(AgentState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("planner", interactive_planner)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "planner")
builder.add_edge("planner", END)

retention_agent = builder.compile()

In [ ]:
# 1. Prepare data from Milestone 1 results
sample_idx = 0
raw_risk = y_prob[sample_idx] * 100
risk_str = f"{round(raw_risk, 2)}%"

# 2. Extract Top 3 Drivers for this specific prediction
importances = xgb.feature_importances_
feature_names = X.columns
top_indices = importances.argsort()[-3:][::-1]
top_features = [feature_names[i] for i in top_indices]

print(f"📊 Analyzing Customer CUST-101...")
print(f"🚩 Top Churn Drivers identified: {', '.join(top_features)}")

# 3. Initialize Agentic State
# (Ensuring all keys match your TypedDict AgentState exactly)
inputs = {
    "customer_id": "CUST-101",
    "churn_risk": risk_str,
    "important_features": top_features,
    "retrieved_strategies": "", # Will be populated by Node 1
    "user_query": "Generate a professional retention report. Explain the logic of each intervention clearly.",
    "chat_history": [],
    "final_recommendation": ""
}

# 4. Invoke the LangGraph Agent
try:
    final_output = retention_agent.invoke(inputs)
    print("\n" + "="*40)
    print("--- 🤖 AGENTIC RETENTION REPORT ---")
    print("="*40 + "\n")
    print(final_output['final_recommendation'])
except Exception as e:
    print(f"❌ Execution Error: {e}")